# ds000245 fMRI Topological Data Analysis Pipeline

**Purpose:** Compute persistent homology (Betti numbers, total persistence) and classical graph metrics on resting-state fMRI connectivity for 45 Parkinson's subjects (15 CTL, 15 ODN, 15 ODP) and perform group-level comparison with effect sizes.

**Runtime:** ~2–3 hours for full 45-subject cohort on Colab CPU. Progress is printed to console.

**Outputs:**
- `cohort_result.json` — per-subject topology + connectivity metrics, group statistics (F, η², p-values)
- Intermediate logs and checkpoints in `outputs/fmri_tda/`


## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted at /content/drive")

## Step 2: Clone the repo and install dependencies

In [ ]:
!cd /content && git clone https://github.com/ChristianV997/ScienceR-Dsim.git
!cd /content/ScienceR-Dsim && git checkout claude/fmri-tda-pipeline
print("✓ Repository cloned and branch checked out")

In [ ]:
# Install scientific dependencies
!pip install -q nibabel nilearn ripser networkx statsmodels scikit-learn scipy numpy pandas matplotlib
print("✓ All dependencies installed")

## Step 3: Configure paths and run the pipeline

In [ ]:
import sys
import os
from pathlib import Path

# Add repo to path
sys.path.insert(0, '/content/ScienceR-Dsim')

# Paths
BIDS_ROOT = Path('/content/drive/MyDrive/OpenNeuro BIDS')
OUT_DIR = Path('/content/drive/MyDrive/fmri_tda_results')
OUT_DIR.mkdir(exist_ok=True)

print(f"BIDS root: {BIDS_ROOT}")
print(f"Output dir: {OUT_DIR}")
print(f"BIDS root exists: {BIDS_ROOT.exists()}")

# List subjects found
subjects = sorted([d.name for d in BIDS_ROOT.glob('sub-*') if d.is_dir() and 'bold' in str(list(d.glob('func/*.nii*')))])
print(f"\n✓ Found {len(subjects)} subjects with BOLD data:")
for s in subjects[:5]:
    print(f"  - {s}")
if len(subjects) > 5:
    print(f"  ... and {len(subjects)-5} more")

In [ ]:
from dual_engine.fmri_tda_pipeline import run_full_analysis

print("\n" + "="*70)
print("STARTING PIPELINE: ds000245 fMRI TDA Analysis")
print("="*70)

results = run_full_analysis(
    bids_root=str(BIDS_ROOT),
    atlas=None,  # Use offline coordinate-based parcellation for robustness
    t_r=2.5,     # ds000245 TR
    n_parcels=100,
    out_dir=str(OUT_DIR),
    verbose=True
)

print("\n" + "="*70)
print("✓ PIPELINE COMPLETE")
print("="*70)

## Step 4: Review results

In [ ]:
import json

result_file = OUT_DIR / 'cohort_result.json'
if result_file.exists():
    with open(result_file) as f:
        res = json.load(f)
    
    print("\nCOHORT RESULT SUMMARY")
    print("="*70)
    print(f"Provenance: {res.get('provenance')}")
    print(f"Subjects processed: {res.get('n_subjects')}")
    print(f"\nGroup-level statistics (one-way ANOVA + effect size η²):")
    print()
    
    for metric in ['total_persistence_h1', 'betti1_mean', 'modularity', 'global_efficiency']:
        stats = res.get('group_stats', {}).get(metric, {})
        if stats:
            print(f"{metric:.<40}")
            print(f"  CTL: {stats.get('CTL_mean', 'N/A'):.3f} ± {stats.get('CTL_std', 0):.3f}")
            print(f"  ODN: {stats.get('ODN_mean', 'N/A'):.3f} ± {stats.get('ODN_std', 0):.3f}")
            print(f"  ODP: {stats.get('ODP_mean', 'N/A'):.3f} ± {stats.get('ODP_std', 0):.3f}")
            print(f"  F = {stats.get('F_value', 'N/A'):.2f}, η² = {stats.get('effect_size_eta2', 'N/A'):.3f}, p = {stats.get('p_value', 'N/A'):.4f}")
            print()
    
    print(f"\n✓ Full results saved to: {result_file}")
else:
    print("⚠ Result file not found. Check pipeline output above for errors.")

## Step 5: Download results (optional)

Uncomment the cell below to download the JSON results to your local machine.

In [ ]:
# from google.colab import files
# files.download(str(result_file))
# print(f"✓ Downloaded {result_file.name}")

## Troubleshooting

**If the pipeline fails:**

1. **"File not found" for BIDS_ROOT:** Check that the path `/content/drive/MyDrive/OpenNeuro BIDS` exists in your Drive. Adjust the path in Step 3 if needed.

2. **"No module named 'nibabel'":** Run the pip install cell again (`pip install nibabel nilearn ...`).

3. **Runtime timeout (>3 hours):** Colab may disconnect after 12 hours of idle time. If the pipeline is still running after 10 hours, save intermediate results and continue in a new notebook.

4. **Memory issues:** Reduce `n_parcels` from 100 to 50 if you hit memory limits.

**For questions or bugs**, see the repo: [github.com/ChristianV997/ScienceR-Dsim](https://github.com/ChristianV997/ScienceR-Dsim)
